# Exploratory Data Analysis (EDA) - South African National Lottery

This notebook contains the Exploratory Data Analysis (EDA) and visualizations for the South African National Lottery datasets (PowerBall, PowerBall Xtra, and Lotto) processed by **ArcStractor**.

## Objectives:
1. **Frequency Analysis**: Explore occurrences of numbers to identify hot and cold numbers.
2. **Sum Distributions**: Analyze draw sums to check for normality.
3. **Variable Correlation**: Explore dependencies between calendar features and draw metrics.
4. **Parity Analysis**: Test observed odd/even splits against binomial projections.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import scipy.stats as stats

# Set theme
sns.set_theme(style="darkgrid")
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

## 1. Load Cleaned Dataset
Choose a dataset to analyze (e.g. `powerball_clean.csv`, `lotto_clean.csv`, `powerball_xtra_clean.csv`).

In [ ]:
dataset_path = "../data/cleaned/powerball_clean.csv"
df = pd.read_csv(dataset_path)
print(f"Loaded {len(df)} draws from {dataset_path}")
df.head()

## 2. Main Number Frequency (Hot vs. Cold Numbers)
We identify the occurrence frequency of all main balls drawn. The expected uniform distribution mean is overlaid.

In [ ]:
ball_cols = [c for c in df.columns if c.startswith("ball_")]
num_main = len(ball_cols)
all_numbers = df[ball_cols].values.flatten()
max_num = int(np.max(all_numbers))

freq_series = pd.Series(all_numbers).value_counts().reindex(range(1, max_num + 1), fill_value=0)
sorted_freqs = freq_series.sort_values(ascending=False)

print("Top 5 Hot Numbers:")
print(sorted_freqs.head(5))
print("\nBottom 5 Cold Numbers:")
print(sorted_freqs.tail(5))

# Plot
plt.figure(figsize=(12, 5))
colors = ['#1f77b4' for _ in range(max_num)]
for idx, num in enumerate(freq_series.index):
    if num in sorted_freqs.head(5).index:
        colors[idx] = '#d62728'  # Red for hot
    elif num in sorted_freqs.tail(5).index:
        colors[idx] = '#bcbd22'  # Yellow for cold

plt.bar(freq_series.index, freq_series.values, color=colors, edgecolor='black', alpha=0.8)
plt.axhline(y=len(all_numbers)/max_num, color='#2ca02c', linestyle='--', label='Expected Uniform Mean')
plt.title("Main Number Frequency Distribution")
plt.xlabel("Ball Number")
plt.ylabel("Occurrences")
plt.xticks(range(1, max_num + 1, 2 if max_num > 40 else 1))
plt.legend()
plt.show()

## 3. Draw Sum Distribution
We plot a histogram of `sum_main_balls` with an overlaid fitted normal curve. According to the Central Limit Theorem, the sum of independent numbers drawn should approximate a normal distribution.

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df['sum_main_balls'], kde=True, color='#9467bd', stat="density", bins=20, alpha=0.6)

mu_sum = np.mean(df['sum_main_balls'])
sigma_sum = np.std(df['sum_main_balls'])
x_range = np.linspace(df['sum_main_balls'].min(), df['sum_main_balls'].max(), 200)
plt.plot(x_range, stats.norm.pdf(x_range, mu_sum, sigma_sum), color='#d62728', linewidth=2, label='Fitted Normal Curve')
plt.title("Draw Sum Distribution with Fitted Normal Curve")
plt.xlabel("Sum of Main Balls")
plt.ylabel("Probability Density")
plt.legend()
plt.show()

## 4. Feature Correlations
Let's check if there are any linear relationships between variables such as calendar features, draw statistics, and parity.

In [ ]:
corr_cols = [
    "year", "month", "day_of_month", "day_of_week", "is_weekend",
    "sum_main_balls", "mean_main_balls", "min_main_ball", "max_main_ball",
    "range_main_balls", "odd_count", "even_count", "is_powerball_even"
]
corr_cols = [c for c in corr_cols if c in df.columns]

plt.figure(figsize=(10, 8))
corr_matrix = df[corr_cols].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, cmap="coolwarm", fmt=".2f", square=True, linewidths=0.5)
plt.title("Analytical Correlation Matrix")
plt.show()

## 5. Odd/Even Splits vs. Binomial Projections
Finally, we compare the observed count of odd balls per draw against the theoretical probabilities of the Binomial Distribution $B(N, 0.5)$.

In [ ]:
obs_counts = df['odd_count'].value_counts().reindex(range(0, num_main + 1), fill_value=0)

expected_freqs = []
for k in range(num_main + 1):
    pmf_val = stats.binom.pmf(k, num_main, 0.5)
    expected_freqs.append(len(df) * pmf_val)

x = np.arange(num_main + 1)
width = 0.35

plt.figure(figsize=(8, 5))
plt.bar(x - width/2, obs_counts.values, width, label='Observed Counts', color='#2ca02c', alpha=0.8)
plt.bar(x + width/2, expected_freqs, width, label='Binomial Expectation B(N, 0.5)', color='#ff7f0e', alpha=0.8)
plt.title("Odd Balls Distribution vs. Binomial Expectations")
plt.xlabel("Odd Balls per Draw")
plt.ylabel("Number of Draws")
plt.xticks(x)
plt.legend()
plt.show()